In [8]:
import ijson
import numpy as np
import json
from decimal import Decimal
import json
import time
import torch
import warnings
import numpy as np
from IPython.display import clear_output
import pennylane as qml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.stats import gaussian_kde

In [ ]:
datos = np.array(datos)
X_train, X_temp = train_test_split(
    datos, 
    train_size=10000, 
    random_state=42, 
    shuffle=True
)

X_val, rest = train_test_split(
    X_temp, 
    train_size=2500, 
    random_state=42, 
    shuffle=True
)

X_inf, rest = train_test_split(
    rest, 
    train_size=10000, 
    random_state=42, 
    shuffle=True
)

# Verificar tamaños
print(f"Entrenamiento: {len(X_train)}")
print(f"Validación: {len(X_val)}")
print(f"Inferencia: {len(X_inf)}")


In [9]:
# Gell-Mann matrices (SU(3))
Lambda = {
    1: torch.tensor([[0, 1, 0],
                 [1, 0, 0],
                 [0, 0, 0]], dtype=torch.cdouble),

    2: torch.tensor([[0, -1j, 0],
                 [1j, 0, 0],
                 [0, 0, 0]], dtype=torch.cdouble),

    3: torch.tensor([[1, 0, 0],
                 [0, -1, 0],
                 [0, 0, 0]], dtype=torch.cdouble),

    4: torch.tensor([[0, 0, 1],
                 [0, 0, 0],
                 [1, 0, 0]], dtype=torch.cdouble),

    5: torch.tensor([[0, 0, -1j],
                 [0, 0, 0],
                 [1j, 0, 0]], dtype=torch.cdouble),

    6: torch.tensor([[0, 0, 0],
                 [0, 0, 1],
                 [0, 1, 0]], dtype=torch.cdouble),

    7: torch.tensor([[0, 0, 0],
                 [0, 0, -1j],
                 [0, 1j, 0]], dtype=torch.cdouble),

    8: (1/torch.sqrt(torch.tensor(3.0))) * torch.tensor([[1, 0, 0],
                                  [0, 1, 0],
                                  [0, 0, -2]], dtype=torch.cdouble),
    0: torch.tensor([[1, 0, 0],
                 [0, 1, 0],
                 [0, 0, 1]], dtype=torch.cdouble)
}

# Spin-1 rotation generators (SO(3) ⊂ SU(3))
Sigma = {
    1: (1 / torch.sqrt(torch.tensor(2.0))) * torch.tensor([
        [0, 1, 0],
        [1, 0, 1],
        [0, 1, 0]
    ], dtype=torch.cdouble),

    2: (1 / torch.sqrt(torch.tensor(2.0))) * torch.tensor([
        [0, -1j, 0],
        [1j, 0, -1j],
        [0, 1j, 0]
    ], dtype=torch.cdouble),

    3: torch.tensor([
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, -1]
    ], dtype=torch.cdouble)
}

def TSWAP_matrix():
    tswap = np.zeros((9, 9), dtype=complex)
    for i in range(3):
        for j in range(3):
            ket = np.zeros(9)
            bra = np.zeros(9)
            ket[3*i + j] = 1   # |i⟩|j⟩
            bra[3*j + i] = 1   # |j⟩|i⟩
            tswap += np.outer(bra, ket)
    return tswap


def unitary_from_generator(generator_matrix, theta):
    if not torch.is_tensor(theta):
        theta = torch.tensor(theta, dtype=torch.cdouble)
    i = torch.tensor(1j, dtype=torch.cdouble)
    return Lambda[0] + (torch.cos(theta) - torch.tensor(1.0)) * generator_matrix @ generator_matrix + i * torch.sin(theta) * generator_matrix

def inicializing_qutrit_state(theta1, theta2, phi1, phi2):
    Gamma= 0
    a0 = 0
    a1 = 0
    a2 = 0

    Gamma = torch.sqrt(torch.tensor(2.0)) * (torch.tensor(3.0) + torch.cos(theta1)*torch.cos(theta2) + torch.sin(theta1)*torch.sin(theta2)*torch.cos(phi1 - phi2))**(torch.tensor(-0.5))

    a0 = (torch.sqrt(torch.tensor(2.0)) * torch.cos(theta1/2) * torch.cos(theta2/2)).item()
    a1 = (torch.exp(1j * phi1) * torch.sin(theta1/2) * torch.cos(theta2/2) + torch.cos(theta1/2) * torch.sin(theta2/2) * torch.exp(1j * phi2)).item()
    a2 = (torch.sqrt(torch.tensor(2.0)) * torch.exp(1j * (phi1 + phi2)) * torch.sin(theta1/2) * torch.sin(theta2/2)).item()

    state = Gamma * torch.tensor([a0, a1, a2], dtype=torch.cdouble)
    state = state / torch.linalg.norm(state)
    
    

    return state.detach().clone().numpy()

import numpy as np
from scipy.linalg import qr

def unitary_from_state(psi):
    """
    psi: complex normalized vector of dimension 3 (qutrit)
    returns: 3x3 unitary whose first column is psi
    """
    psi = psi / np.linalg.norm(psi)  # por seguridad
    
    a1 = torch.tensor([0.555,0, 0.555], dtype=torch.cdouble)
    a2 = torch.tensor([0.555,0.555, 0], dtype=torch.cdouble)
    mat = np.column_stack([psi, a1 , a2])

    Q, R = qr(mat)
    phase = np.vdot(psi, Q[:,0])
    Q[:,0] = Q[:,0] * (phase/abs(phase)).conj()
    
    return Q

In [ ]:
from autoray import do

num_particles = 8
wires = list(range(num_particles))  # +1 ancilla
ancilla = wires[0]
dev = qml.device("default.qutrit", wires=wires)  
unitaries = []

# QAE circuit for qutrits
@qml.qnode(dev, interface="torch", diff_method="backprop")
def qae_circuit_qutrit(jets, w, theta_i, phi_i, w_i, num_layers, target_prob):
    unitaries = []
    encoding_Majorana(jets, w, unitaries)
    variational_layer_qutrit(theta_i, phi_i, w_i, num_layers)
    return qml.expval(Sigma[3], wires=ancilla), target_prob

# Encoding for qutrits
def encoding_Majorana(jets, w, unitaries):

    #Ejemplo mientras no tengo información sobre el dataset
    theta = jets['theta']
    phi = jets['phi']
    d0 = jets['d0']
    dz = jets['dz']

        initial_state = inicializing_qutrit_state( theta, phi, d0, dz)
        u = unitary_from_state(initial_state)
        unitaries.append(u)
        qml.QutritUnitary(u, wires=i)    

# Variational layer for qutrits 
def variational_layer_qutrit(theta_i, phi_i, w_i, num_layers):
    for layer in range(num_layers):
        for i in range(num_particles):

            RX = unitary_from_generator(Sigma[1], phi_i[layer, i])
            RY = unitary_from_generator(Sigma[2], theta_i[layer, i])
            RZ = unitary_from_generator(Sigma[3], w_i[layer, i])
    
            qml.QutritUnitary(RX, wires=i)
            qml.QutritUnitary(RZ, wires=i)
            qml.QutritUnitary(RY, wires=i)

        for i in range(num_particles):
            if i == num_particles - 1:
                qml.TAdd(wires=[i, 0])
            else:
                for j in range(i + 1, num_particles):
                    qml.TAdd(wires=[i, j])

### **Mean Squared Error Loss**

The trainning process aims to find the values of the model parameters $\theta$ that minimise the Mean Squared Error loss function,
$$
\mathbb{L}(\theta) = \frac{1}{N}\sum_{i=1}^N (P_{b}^i(\theta) - T^i)^2 
$$
 where $N$ is the number of trainning jets, $P_{b}^{i}$ is the rpedicted probability for the $i$-th jet, and $T^{i}$ is the target probability for the $i$-th jet, $i.e.$, 1 for a $b$ jet and 0 for a $\bar{b}$ jet.
 {\color{red} En nuestro caso voy a escalar las ecuaciones a qutrits}
$$
P_b = \frac{1}{3}(\langle \Sigma_{3}^0 \rangle + 1)
$$
$$
P_{\bar{b}} = \frac{1}{3}(1 - \langle \Sigma_{3}^0 \rangle) = 1 - P_b
$$

In [ ]:
#Mean squared error cost function
def cost_function(jet, w, theta_i, phi_i, w_i, num_layers, trainning_jets, target_prob):
    N = trainning_jets
    temp = 0
    val_0 = qae_circuit_qutrit(jet, w, theta_i, phi_i, w_i, num_layers)
    prob_b = 1/3 * ( val_0 + 1 )
    for i in range(N):
        temp += (prob_b - target_prob)**2
    return (1/N) * temp


# **Training step**

Due to the large number of jets in the datasets, the quanutm models are trained implementing a mini-batch gradient descent algorithim using the Adam optimizer to minimize:
$$
\mathbb{L}(\theta) = \frac{1}{N}\sum_{i=1}^N (P_{b}^i(\theta) - T^i)^2. 
$$
The trainning dataset is split in several mini-batches containning a fixed number of trinning samples. During each trainning step, the gradient of the last equation is evaluated averaging over the trainning samples of a mini-batch, and used to update model aprameters. A trainning epoch is completed when the whole trainning dataset is processed, namely after a number of steps equal to the number of mini-batches.

Unless specified otherwise, the models are trained with learning rate $\xi = 0.01$ for 100 epochs, while the minibatch size is fixed to the maximun value allowed by memory constraints. The output of the classifier gives the probability tahat a jetis generated by a $b$ or a $\bar{b}$ quark. The label with the highest probability is assigned to the jet, $i.e.$ if $P_b > 0.5 (P_b > 0.5)$ then is classified as a $b$ jet ($\bar{b}$ jet).

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
num_layers = 1 # Number of variational layers
theta_i = (torch.rand(num_layers, num_particles) * 2 * torch.pi).requires_grad_(True)
phi_i   = (torch.rand(num_layers, num_particles) * 2 * torch.pi).requires_grad_(True)
w_i     = (torch.rand(num_layers, num_particles) * 2 * torch.pi).requires_grad_(True)

optimizer = torch.optim.Adam(
    [w, theta_i, phi_i, w_i],
    lr=0.01,              
    betas=(0.5, 0.999),
    eps=1e-08,
    weight_decay=0.0,    
    amsgrad=True          
)
num_epochs = 100
all_fidelities = []
event_fidelities = []  # List to store event fidelities

inicio = time.time()
# Training 
for epoch in range(num_epochs):
    total_loss = 0.0
    epoch_fidelities = []
    avg_fidelity = 0.0
    avg_loss = 0.0
    
    for jet in X_train:
        if len(jet['constituents']) < num_particles:
            continue
    
        loss, fidelity = cost_function(jet, w, theta_i, phi_i, w_i,  num_layers)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        epoch_fidelities.append(fidelity)
        event_fidelities.append(fidelity * 100)  # in %

    avg_loss = total_loss / len(epoch_fidelities)
    avg_fidelity = np.mean(epoch_fidelities) * 100
    all_fidelities.append(avg_fidelity)
    
    print(f"Epoch {epoch+1}, Loss: {1 + avg_loss}, Avg Fidelity: {avg_fidelity}%")

end = time.time()

tiempo_entrenamiento = (end - inicio)/ 60